In [ ]:
import os

# 设置 Hugging Face 镜像
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import huggingface_hub.constants as hf_constants

hf_constants.ENDPOINT = os.environ["HF_ENDPOINT"]

print("HF_ENDPOINT =", os.environ["HF_ENDPOINT"])

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "amazon/chronos-2"

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE
)

print("Chronos-2 loaded successfully!")

In [ ]:
DATA_PATH = "../data/20260416114320889723788079931057.csv"  # 真实停车场数据（长数字文件名）

# id / ploid 都是超长数字，必须按字符串读取，否则会被转成科学计数法
raw = pd.read_csv(DATA_PATH, dtype={"id": str, "ploid": str})

# 用可读时间列解析时间戳（createtime 是毫秒，datestring 是 yyyy-MM-dd HH:mm:ss）
raw["timestamp"] = pd.to_datetime(raw["datestring"])

# total = 总车位数（恒定 792），rest = 剩余车位数
# occupied = 已占用车位数，occupancy_rate = 占用率
raw["occupied"] = raw["total"] - raw["rest"]
raw["occupancy_rate"] = raw["occupied"] / raw["total"]

# 按时间排序，同一时间戳保留最后一条
raw = raw.sort_values("timestamp").reset_index(drop=True)
raw = raw.drop_duplicates(subset=["timestamp"], keep="last")

# 原始数据约 2 分钟一条，重采样到 5 分钟均值（与 parking.csv 口径一致）
ts = raw.set_index("timestamp")["occupancy_rate"]
ts = ts.resample("5min").mean().dropna()

series_df = ts.reset_index()  # columns: timestamp, occupancy_rate

print(f"原始行数: {len(raw)}")
print(f"重采样(5min)后行数: {len(series_df)}")
print(f"时间范围: {series_df['timestamp'].min()} ~ {series_df['timestamp'].max()}")
print(f"总车位数: 792（total 恒定值）")
display(series_df.head())

In [ ]:
series_df["occupancy_rate"] = pd.to_numeric(series_df["occupancy_rate"], errors="coerce")
series_df = series_df.dropna().sort_values("timestamp").reset_index(drop=True)

print("缺失值:", series_df.isna().sum().to_dict())
print("占用率 Min/Max:", round(series_df["occupancy_rate"].min(), 4), round(series_df["occupancy_rate"].max(), 4))
print("占用率 Mean:", round(series_df["occupancy_rate"].mean(), 4))

if not series_df["occupancy_rate"].between(0, 1).all():
    print("Warning: 占用率存在 0~1 范围外的值，请确认。")

display(series_df[["timestamp", "occupancy_rate"]].describe())

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(series_df["timestamp"], series_df["occupancy_rate"], linewidth=0.6, color="steelblue")
plt.title("停车场占用率历史序列（2026-09-17 ~ 2026-11-04）")
plt.xlabel("时间")
plt.ylabel("占用率")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 放大最后 3 天
plt.figure(figsize=(16, 4))
plt.plot(series_df["timestamp"].tail(3 * 288), series_df["occupancy_rate"].tail(3 * 288), linewidth=0.8, color="steelblue")
plt.title("近 3 天占用率")
plt.xlabel("时间")
plt.ylabel("占用率")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-8, denom)
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)


def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1e-8
    if not mask.any():
        return np.nan
    return 100.0 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def wql(y_true, y_pred, q):
    # 分位数损失（pinball loss），q=0.5 时为 MAE/2
    err = y_true - y_pred
    return np.mean(np.maximum(q * err, (q - 1.0) * err))


def coverage(y_true, lower, upper):
    return float(np.mean((y_true >= lower) & (y_true <= upper)))


def run_backtest(train, test, horizon):
    context_df = pd.DataFrame({
        "id": "parking_real_001",
        "timestamp": train["timestamp"],
        "target": train["occupancy_rate"],
    })

    pred = pipeline.predict_df(
        context_df,
        prediction_length=horizon,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
    )

    forecast = pred[["timestamp", "0.1", "0.5", "0.9"]].copy()
    forecast.columns = ["timestamp", "q10", "q50", "q90"]

    eval_df = test[["timestamp", "occupancy_rate"]].merge(forecast, on="timestamp", how="inner")
    if len(eval_df) != horizon:
        print(f"Warning: 匹配数量 {len(eval_df)} != {horizon}")

    return eval_df


print("指标函数与回测函数定义完成")

In [ ]:
PREDICTION_LENGTH = 12  # 5 分钟一个点，12 点 = 1 小时

train_df = series_df.iloc[:-PREDICTION_LENGTH].copy()
test_df = series_df.iloc[-PREDICTION_LENGTH:].copy()

print("训练点数:", len(train_df), "测试点数:", len(test_df))

eval_df = run_backtest(train_df, test_df, PREDICTION_LENGTH)

y_true = eval_df["occupancy_rate"].to_numpy()
y_pred = eval_df["q50"].to_numpy()
y_low = eval_df["q10"].to_numpy()
y_high = eval_df["q90"].to_numpy()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape_v = mape(y_true, y_pred)
smape_v = smape(y_true, y_pred)
wql_v = wql(y_true, y_pred, 0.5)
cov = coverage(y_true, y_low, y_high)

metrics = pd.DataFrame([{
    "Model": "Chronos-2",
    "Horizon(points)": PREDICTION_LENGTH,
    "MAE": mae,
    "RMSE": rmse,
    "MAPE(%)": mape_v,
    "sMAPE(%)": smape_v,
    "WQL(0.5)": wql_v,
    "Coverage(80%)": cov,
}])

print("========== Chronos-2 回测结果（预测未来 1 小时） ==========")
display(metrics)

eval_df.to_csv("chronos_real_backtest_predictions.csv", index=False, encoding="utf-8-sig")
metrics.to_csv("chronos_real_backtest_metrics.csv", index=False, encoding="utf-8-sig")
print("已保存: chronos_real_backtest_predictions.csv / chronos_real_backtest_metrics.csv")

In [ ]:
horizons = [12, 24, 48, 96]  # 1h / 2h / 4h / 8h（5 分钟一个点）

rows = []
for h in horizons:
    tr = series_df.iloc[:-h].copy()
    te = series_df.iloc[-h:].copy()
    e = run_backtest(tr, te, h)
    yt = e["occupancy_rate"].to_numpy()
    yp = e["q50"].to_numpy()
    rows.append({
        "Horizon(points)": h,
        "Horizon(hours)": h * 5 / 60,
        "MAE": mean_absolute_error(yt, yp),
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "MAPE(%)": mape(yt, yp),
        "sMAPE(%)": smape(yt, yp),
        "Coverage(80%)": coverage(yt, e["q10"].to_numpy(), e["q90"].to_numpy()),
    })

multi = pd.DataFrame(rows)
print("========== 多步长回测对比 ==========")
display(multi)
multi.to_csv("chronos_real_multihorizon_metrics.csv", index=False, encoding="utf-8-sig")

In [ ]:
context_tail = train_df.tail(200)

plt.figure(figsize=(14, 5))
plt.plot(context_tail["timestamp"], context_tail["occupancy_rate"], color="gray", linewidth=1, label="历史（前 200 点）")
plt.plot(eval_df["timestamp"], eval_df["occupancy_rate"], marker="o", label="实际占用率")
plt.plot(eval_df["timestamp"], eval_df["q50"], marker="s", linestyle="--", label="Chronos-2 预测（中位数）")
plt.fill_between(eval_df["timestamp"], eval_df["q10"], eval_df["q90"], color="orange", alpha=0.2, label="80% 预测区间")
plt.title("停车占用率预测 vs 实际（测试集最后 12 点 = 1 小时）")
plt.xlabel("时间")
plt.ylabel("占用率")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
resid = y_true - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(y_pred, y_true, alpha=0.7)
axes[0].plot([0, 1], [0, 1], "r--", linewidth=1)
axes[0].set_xlabel("预测值")
axes[0].set_ylabel("实际值")
axes[0].set_title("预测 vs 实际散点")
axes[0].grid(alpha=0.3)

axes[1].plot(eval_df["timestamp"], resid, marker="o")
axes[1].axhline(0, color="red", linewidth=1)
axes[1].set_xlabel("时间")
axes[1].set_ylabel("残差（实际 - 预测）")
axes[1].set_title("残差时间序列")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()